# Retail Analytics Platform

## Phase 3 - Customer Segmentation (RFM Analysis)

### Objective
Perform customer segmentation using the RFM methodology.

### RFM Components

- Recency
- Frequency
- Monetary Value

### Output

- Customer Scores
- Customer Segments
- Business Recommendations

### Author

Umesh Zure

In [2]:
# --------------------------------------------------
# Retail Analytics Platform
# Phase 3 - Customer Segmentation
# --------------------------------------------------

import pandas as pd
import numpy as np

print("Libraries loaded successfully")

Libraries loaded successfully


In [3]:
df = pd.read_csv("../data/sample_data/retail_sales.csv")

df.head()

,Order_ID,Order_Date,Store,Category,Product,Revenue,Quantity
0,1001,2026-01-10,Mumbai,Electronics,Laptop,85000,2
1,1002,2026-01-11,Pune,Furniture,Desk,25000,5
2,1003,2026-01-12,Delhi,Electronics,Mobile,65000,10
3,1004,2026-01-13,Bangalore,Office Supplies,Printer,15000,3
4,1005,2026-01-14,Mumbai,Furniture,Chair,12000,8


In [4]:
customer_ids = [
    "CUST001",
    "CUST002",
    "CUST003",
    "CUST001",
    "CUST004",
    "CUST005",
    "CUST002",
    "CUST003",
    "CUST006",
    "CUST001"
]

df["Customer_ID"] = customer_ids

df.head()

,Order_ID,Order_Date,Store,Category,Product,Revenue,Quantity,Customer_ID
0,1001,2026-01-10,Mumbai,Electronics,Laptop,85000,2,CUST001
1,1002,2026-01-11,Pune,Furniture,Desk,25000,5,CUST002
2,1003,2026-01-12,Delhi,Electronics,Mobile,65000,10,CUST003
3,1004,2026-01-13,Bangalore,Office Supplies,Printer,15000,3,CUST001
4,1005,2026-01-14,Mumbai,Furniture,Chair,12000,8,CUST004


In [5]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"])

print("Date conversion completed")

Date conversion completed


In [6]:
analysis_date = df["Order_Date"].max()

print("Analysis Date:", analysis_date)

Analysis Date: 2026-01-19 00:00:00


In [7]:
rfm = df.groupby("Customer_ID").agg({
    "Order_Date": lambda x: (analysis_date - x.max()).days,
    "Order_ID": "count",
    "Revenue": "sum"
})

rfm.columns = [
    "Recency",
    "Frequency",
    "Monetary"
]

rfm

,Recency,Frequency,Monetary
Customer_ID,,,
CUST001,0,3,122000
CUST002,3,2,30000
CUST003,2,2,93000
CUST004,5,1,12000
CUST005,4,1,35000
CUST006,1,1,4000


In [8]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    3,
    labels=[3, 2, 1]
)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    3,
    labels=[1, 2, 3]
)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    3,
    labels=[1, 2, 3]
)

rfm

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
Customer_ID,,,,,,
CUST001,0,3,122000,3,3,3
CUST002,3,2,30000,2,2,2
CUST003,2,2,93000,2,3,3
CUST004,5,1,12000,1,1,1
CUST005,4,1,35000,1,1,2
CUST006,1,1,4000,3,2,1


In [9]:
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str)
    + rfm["F_Score"].astype(str)
    + rfm["M_Score"].astype(str)
)

rfm

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
Customer_ID,,,,,,,
CUST001,0,3,122000,3,3,3,333
CUST002,3,2,30000,2,2,2,222
CUST003,2,2,93000,2,3,3,233
CUST004,5,1,12000,1,1,1,111
CUST005,4,1,35000,1,1,2,112
CUST006,1,1,4000,3,2,1,321


In [10]:
def segment_customer(score):

    if score >= "333":
        return "Champions"

    elif score >= "233":
        return "Loyal Customers"

    elif score >= "132":
        return "Potential Loyalists"

    else:
        return "At Risk"

rfm["Segment"] = rfm["RFM_Score"].apply(segment_customer)

rfm

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
Customer_ID,,,,,,,,
CUST001,0,3,122000,3,3,3,333,Champions
CUST002,3,2,30000,2,2,2,222,Potential Loyalists
CUST003,2,2,93000,2,3,3,233,Loyal Customers
CUST004,5,1,12000,1,1,1,111,At Risk
CUST005,4,1,35000,1,1,2,112,At Risk
CUST006,1,1,4000,3,2,1,321,Loyal Customers


In [11]:
segment_summary = (
    rfm["Segment"]
    .value_counts()
    .reset_index()
)

segment_summary

,Segment,count
0,Loyal Customers,2
1,At Risk,2
2,Champions,1
3,Potential Loyalists,1


In [12]:
rfm.to_csv(
    "../data/processed/rfm_customer_segmentation.csv"
)

segment_summary.to_csv(
    "../data/processed/rfm_segment_summary.csv"
)

print("RFM outputs exported successfully")

RFM outputs exported successfully
